In [ ]:
#enable autoreload

%load_ext autoreload
%autoreload all

#common jupyter functions
from IPython.display import display, clear_output

import sys, os.path
importPath = os.path.abspath('../../common')
if not importPath in sys.path:
    sys.path.append(importPath)

In [ ]:
#define training logs path
trainingLogsPath = f"../logs/potion_mtclmtlbl_words_v2_mergedlbls/"
print(trainingLogsPath)

#define path for data files
baseDataPath = "./data"
dataFileSuffix = "-merged-lbls"
print(baseDataPath)

#define dataset source path
dsetPath = "../data/preproc/ktu/corpus_AICP_FIMI_2026-01-24-merged-lbls.ranged.json"

#define sequence length for the model (this is also used in tokenizer)
modelSeqLen = 1024

In [ ]:
# #(re)build the dataset

# #import dependencies
# from corpus.hdf5Dset import MultilabelDataset, MultilabelDatasetBuilder, MultilabelDatasetSplitter, MultilabelDatasetPermutator
# from corpus.chunkedCorpus import ChunkedCorpus
# from random import Random
# import torch
# import numpy as np

# #load corpus
# corpus = ChunkedCorpus.loadFromJson(dsetPath)
# # corpus.texts.sort(key=lambda x: len(x.text), reverse=True)
# # corpus.texts = corpus.texts[:10]

# # load tokenizer
# from nns.potion_multiclass_multilabel_v2 import Tokenizer
# tokenizer = Tokenizer.load(modelSeqLen)

# #build main dataset
# from nns.embedders import buildPotionEmbedderV2
# embedder = buildPotionEmbedderV2(torch.device("cpu"))

# #... assume first label of each group is neutral
# lblsForSpecialTkns = np.roll(np.array(corpus.labelGrps, np.int32).cumsum(), 1)
# lblsForSpecialTkns[0] = 0
# print(f"Labels for special tokens are: {lblsForSpecialTkns}")

# #...
# dsetAll = MultilabelDataset.create(
#     os.path.join(baseDataPath, f"dataset-words-all{dataFileSuffix}"),
#     maxSeqLen = modelSeqLen,
#     embedDimLen = 256,
#     numLbls = sum(corpus.labelGrps)
# )
# dsb = MultilabelDatasetBuilder(tokenizer=tokenizer, embedder=embedder, corpus=corpus)
# dsb.buildWordSamples(dset = dsetAll, lblsForSpecialTkns=lblsForSpecialTkns)
# dsetAll.saveToFiles()
# print(f"Sample count all: {dsetAll.textIds.shape[0]}")

# #create train-validate and test datasets
# dsetAll = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-all{dataFileSuffix}"))

# dss = MultilabelDatasetSplitter()
# dsetTrainValidate, dsetTest = dss.split(
#     dset=dsetAll, 
#     foldDist=[0.9, 0.1], 
#     foldBackingFileNamePrefixes=[
#         os.path.join(baseDataPath, f"dataset-words-train-validate{dataFileSuffix}"),
#         os.path.join(baseDataPath, f"dataset-words-test{dataFileSuffix}")
#     ],
#     rng=Random(0)
# )
# dsetTrainValidate.saveToFiles()
# dsetTest.saveToFiles()

# print(f"Sample count train/validate: {dsetTrainValidate.textIds.shape[0]}")
# print(f"Sample count test: {dsetTest.textIds.shape[0]}")

# del dsetAll

# #create train and validate datasets
# dsetTrainValidate = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-train-validate{dataFileSuffix}"))

# dss = MultilabelDatasetSplitter()
# dsetTrain, dsetValidate = dss.split(
#     dset=dsetTrainValidate, 
#     foldDist=[0.9, 0.1], 
#     foldBackingFileNamePrefixes=[
#         os.path.join(baseDataPath, f"dataset-words-train{dataFileSuffix}"),
#         os.path.join(baseDataPath, f"dataset-words-validate{dataFileSuffix}")
#     ],
#     rng=Random(0)
# )
# dsetTrain.saveToFiles()
# dsetValidate.saveToFiles()

# print(f"Sample count train: {dsetTrain.textIds.shape[0]}")
# print(f"Sample count validate: {dsetValidate.textIds.shape[0]}")

# del dsetTrainValidate, dsetTrain, dsetValidate

# #initialize permutation indices
# dsp = MultilabelDatasetPermutator()

# dsetTrain = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-train{dataFileSuffix}"))
# dsp.initializeIndices(dsetTrain)
# dsetTrain.saveToFiles()
# del dsetTrain

# dsetValidate = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-validate{dataFileSuffix}"))
# dsp.initializeIndices(dsetValidate)
# dsetValidate.saveToFiles()
# del dsetValidate

# dsetTest = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-test{dataFileSuffix}"))
# dsp.initializeIndices(dsetTest)
# dsetTest.saveToFiles()
# del dsetTest

# #remove intermediate files
# MultilabelDataset.deleteFiles(os.path.join(baseDataPath, f"dataset-words-all{dataFileSuffix}"))
# MultilabelDataset.deleteFiles(os.path.join(baseDataPath, f"dataset-words-train-validate{dataFileSuffix}"))

In [ ]:
#initialize empty model
from nns.potion_multiclass_multilabel_v2 import MultiClassMultiLabelTokenLabelerLit
model : MultiClassMultiLabelTokenLabelerLit = None

#load train and validation datasets
from corpus.hdf5Dset import MultilabelDataset

mlDsetTrain = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-train{dataFileSuffix}"))
print(f"Sample count in train dataset: {mlDsetTrain.currentIndex.shape[0]}")

mlDsetValidate = MultilabelDataset.loadFromFiles(os.path.join(baseDataPath, f"dataset-words-validate{dataFileSuffix}"))
print(f"Sample count in validate dataset: {mlDsetValidate.currentIndex.shape[0]}")

#define split index
splitIdx = -1

#load label configuration for the model from the source corpus
from corpus.chunkedCorpus import ChunkedCorpus
corpus = ChunkedCorpus.loadFromJson(dsetPath)

numLabels = corpus.labelGrps
del corpus

print(numLabels)

In [ ]:
#train the model

import torch
import lightning as lit
import lightning.pytorch.callbacks as litcbacks

from nns.utils import LitUtils

from corpus.hdf5Dset import MultilabelTorchDataset, MultilabelTorchDataLoader, MultilabelDatasetPermutator

tdev = torch.device('cuda') 
lu = LitUtils(trainingLogsPath)

#this indicates if code should be configured for debugging
debug = False
disableTorchCompile = True

#define parameters of training loop, each run spawns a separate version in logs
numRuns = 32
numEpochsPerRun = 10

#load or initialize the model if needed
if model is None:
	#try loading model from the checkpoint of the last version
	lastVer = lu.getLastVersion()

	if lastVer is not None:
		#load the model
		lastVerCptPath = lu.getVersionCheckpointPath(lastVer)
		if lastVerCptPath is not None:
			#load the model
			model = MultiClassMultiLabelTokenLabelerLit.load_from_checkpoint(lastVerCptPath)
			model.to(tdev)

			#set dataset split index corresponding to total last epoch relative to last version
			splitIdx = lu.getLastEpoch(lastVer, assumeEpochsPerVersion=numEpochsPerRun)

	#loading failed, initialize fresh model
	if model is None:
		#make model initialization state repeatable
		torch.manual_seed(0)
		
		#
		model = MultiClassMultiLabelTokenLabelerLit(
			numLabels=numLabels,
			numAdditionalEncoderLayers=4,
			seqLen=modelSeqLen
		)
		model.to(tdev)

#create torch datasets and dataloaders
dsTrain = MultilabelTorchDataset(mlDsetTrain)
dsValidate = MultilabelTorchDataset(mlDsetValidate)

dlTrain = MultilabelTorchDataLoader(dsTrain, batchSize=16, debug=debug)
dlValidate = MultilabelTorchDataLoader(dsValidate, debug=debug)

#define a callback to reshuffle the training dataset every epoch
class PrepareForTrainEpochCback(litcbacks.Callback):
	def on_train_epoch_start(self, trainer, pl_module):
		global splitIdx

		#build next shuffle of the train dataset
		dsp = MultilabelDatasetPermutator()
		dsp.buildSpecificShuffle(mlDsetTrain, splitIdx + 1)
		splitIdx += 1

#this is for adaptive gradient clipping
from zclip import ZClipLightningCallback

#run the training loops
for runIdx in range(numRuns):
	#cleanup cell output a bit
	clear_output()

	#disable torch compiler when debugging, otherwise vscode debugger will crash due to multiple python processes getting spawned
	if debug or disableTorchCompile:
		stance = torch.compiler.set_stance("force_eager")
	
	#
	try:
		trainer = lit.Trainer(					
			default_root_dir=trainingLogsPath,
			callbacks=[PrepareForTrainEpochCback(), ZClipLightningCallback(alpha=0.97, z_thresh=2.5)],
			log_every_n_steps = 1 if debug else 50,
					
			max_epochs=numEpochsPerRun,
			
			limit_train_batches = 100 if debug else None,
			limit_val_batches = 100 if debug else None,
			limit_test_batches =100 if debug else None
		)
		trainer.fit(model=model, train_dataloaders=dlTrain, val_dataloaders=dlValidate)	
	finally:
		#restore default compiler stance, if it was modified
		if debug or disableTorchCompile:
			torch.compiler.set_stance("default")